# CS404 Homework 2 - Local Search for N-Queens Problem

**Student:** Eralp Kiziloglu
**Date:** November 15, 2025  

In [1]:
# Importing required libraries
import random
import time
import numpy as np

## Helper Function: Conflict Calculator

This function counts attacking queen pairs on the board. Our representation uses a list where index = column and value = row position of the queen.

In [2]:
def calculateNumberOfConflicts(board):
    """
    returns number of conflicts
    since each queen is in one column, no vertical conflicts
    """
    numberOfConflicts = 0  # initialize as 0
    
    for i in range(0, len(board)):
        for j in range(i + 1, len(board)):
            if board[i] == board[j]:
                numberOfConflicts += 1  # increment if horizontal conflict
            elif abs(i - j) == abs(board[i] - board[j]):
                numberOfConflicts += 1  # increment if diagonal conflict
                
    return numberOfConflicts

## Additional Helper Functions

In [3]:
# Generate a random initial state
def initialize_board(n):
    """Creates random board where each queen is placed randomly in its column"""
    return [random.randint(0, n-1) for _ in range(n)]

# Get all possible neighbor states
def get_neighbors(board):
    """Generate all neighbors by moving each queen to different rows in same column"""
    neighbors = []
    n = len(board)
    
    for col in range(n):
        current_row = board[col]
        for row in range(n):
            if row != current_row:
                # Create neighbor by moving queen in column 'col' to row 'row'
                neighbor = list(board)
                neighbor[col] = row
                neighbors.append(neighbor)
    
    return neighbors

## Part (a): Basic Hill Climbing Implementation

This is the standard hill climbing that picks the best neighbor at each step. It stops when no better neighbor exists or solution is found.

In [4]:
def basic_hill_climbing(n, max_steps=5000):
    """
    Basic hill climbing: always move to the best neighbor
    Returns: (found_solution, num_steps)
    """
    current_state = initialize_board(n)
    current_score = calculateNumberOfConflicts(current_state)
    
    steps = 0
    
    while steps < max_steps:
        steps += 1
        
        # Check if we found solution
        if current_score == 0:
            return True, steps
        
        # Generate all neighbors and find the best one
        neighbors = get_neighbors(current_state)
        best_neighbor = None
        best_score = current_score
        
        for neighbor in neighbors:
            score = calculateNumberOfConflicts(neighbor)
            if score < best_score:
                best_neighbor = neighbor
                best_score = score
        
        # If no improvement possible, we're stuck
        if best_neighbor is None:
            return False, steps
        
        # Move to better state
        current_state = best_neighbor
        current_score = best_score
    
    return False, steps

## Part (b): Hill Climbing with Random Restarts

When basic hill climbing gets stuck, this version restarts from a new random state. Parameter k controls how many restarts to attempt.

In [5]:
def hill_climbing_with_restarts(n, k, max_steps_per_attempt=5000):
    """
    Hill climbing with k random restarts
    Returns: (found_solution, total_steps_across_all_restarts)
    """
    total_steps = 0
    
    for attempt in range(k):
        found, steps = basic_hill_climbing(n, max_steps_per_attempt)
        total_steps += steps
        
        if found:
            return True, total_steps
    
    # All k attempts failed
    return False, total_steps

## Part (c): Stochastic Hill Climbing

Instead of always picking the absolute best neighbor, this randomly chooses among neighbors that improve the current state. Adds some randomness to escape certain local optima.

In [6]:
def stochastic_hill_climbing(n, max_steps=5000):
    """
    Stochastic hill climbing: randomly pick among improving neighbors
    Returns: (found_solution, num_steps)
    """
    current_state = initialize_board(n)
    current_score = calculateNumberOfConflicts(current_state)
    
    steps = 0
    
    while steps < max_steps:
        steps += 1
        
        # Solution found
        if current_score == 0:
            return True, steps
        
        # Find all neighbors that improve current state
        neighbors = get_neighbors(current_state)
        better_neighbors = []
        
        for neighbor in neighbors:
            score = calculateNumberOfConflicts(neighbor)
            if score < current_score:
                better_neighbors.append((neighbor, score))
        
        # No improving neighbors means we're stuck
        if len(better_neighbors) == 0:
            return False, steps
        
        # Randomly pick one improving neighbor
        chosen = random.choice(better_neighbors)
        current_state = chosen[0]
        current_score = chosen[1]
    
    return False, steps

## Running the Experiments

Now I'll run 100 trials for each algorithm and record success rates and timing.

In [7]:
# Configuration
N = 20
NUM_TRIALS = 100

# Seed for reproducibility during testing - remove this line for actual random runs
random.seed(42)

print(f"Starting experiments with N={N}...\n")

# Part (a): Basic Hill Climbing
print("="*40)
print("Testing Basic Hill Climbing...")
start = time.time()
basic_successes = 0
for trial in range(NUM_TRIALS):
    success, _ = basic_hill_climbing(N)
    if success:
        basic_successes += 1
basic_time = time.time() - start
basic_percentage = (basic_successes / NUM_TRIALS) * 100
print(f"Success rate: {basic_successes}/{NUM_TRIALS}")
print(f"Total time: {basic_time:.3f} seconds\n")

# Part (b): Random Restart with k=10
print("="*40)
print("Testing Random Restart (k=10)...")
start = time.time()
restart10_successes = 0
for trial in range(NUM_TRIALS):
    success, _ = hill_climbing_with_restarts(N, k=10)
    if success:
        restart10_successes += 1
restart10_time = time.time() - start
restart10_percentage = (restart10_successes / NUM_TRIALS) * 100
print(f"Success rate: {restart10_successes}/{NUM_TRIALS}")
print(f"Total time: {restart10_time:.3f} seconds\n")

# Part (b): Random Restart with k=100
print("="*40)
print("Testing Random Restart (k=100)...")
start = time.time()
restart100_successes = 0
for trial in range(NUM_TRIALS):
    success, _ = hill_climbing_with_restarts(N, k=100)
    if success:
        restart100_successes += 1
restart100_time = time.time() - start
restart100_percentage = (restart100_successes / NUM_TRIALS) * 100
print(f"Success rate: {restart100_successes}/{NUM_TRIALS}")
print(f"Total time: {restart100_time:.3f} seconds\n")

# Part (c): Stochastic Hill Climbing
print("="*40)
print("Testing Stochastic Hill Climbing...")
start = time.time()
stochastic_successes = 0
for trial in range(NUM_TRIALS):
    success, _ = stochastic_hill_climbing(N)
    if success:
        stochastic_successes += 1
stochastic_time = time.time() - start
stochastic_percentage = (stochastic_successes / NUM_TRIALS) * 100
print(f"Success rate: {stochastic_successes}/{NUM_TRIALS}")
print(f"Total time: {stochastic_time:.3f} seconds\n")

# Summary
print("="*40)
print("FINAL RESULTS:")
print(f"Basic HC: {basic_percentage:.0f}% success in {basic_time:.3f}s")
print(f"Random Restart k=10: {restart10_percentage:.0f}% success in {restart10_time:.3f}s")
print(f"Random Restart k=100: {restart100_percentage:.0f}% success in {restart100_time:.3f}s")
print(f"Stochastic HC: {stochastic_percentage:.0f}% success in {stochastic_time:.3f}s")

Starting experiments with N=20...

Testing Basic Hill Climbing...
Success rate: 1/100
Total time: 7.826 seconds

Testing Random Restart (k=10)...
Success rate: 20/100
Total time: 95.833 seconds

Testing Random Restart (k=100)...
Success rate: 89/100
Total time: 519.520 seconds

Testing Stochastic Hill Climbing...
Success rate: 3/100
Total time: 18.151 seconds

FINAL RESULTS:
Basic HC: 1% success in 7.826s
Random Restart k=10: 20% success in 95.833s
Random Restart k=100: 89% success in 519.520s
Stochastic HC: 3% success in 18.151s


## Analysis

Looking at the results, random restart with k=100 clearly performs best with 89% success rate, though it takes significantly longer (over 8 minutes). The basic and stochastic versions both struggle badly - only 1% and 3% success rates respectively - because they get stuck in local optima and have no mechanism to escape. Random restart with k=10 shows improvement over basic methods but still only achieves 20% success, suggesting that 10 restarts often isn't enough for N=20. Interestingly, stochastic doesn't help much compared to basic, probably because the randomness in neighbor selection isn't enough to consistently escape the deep local optima that exist in this problem space.